In [72]:
import io
import zipfile
import pandas as pd
import requests

def extract_indicator(code, metric_name):
    url = "https://eias.cisstat.org/downloads/api/dataset/download/indicator"
    payload = {"lang": "ru", "format": "csv", "code": [code]}

    res = requests.post(url, json=payload)

    # распаковка архива в памяти
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        fact_file = [f for f in z.namelist() if f.startswith("ts_")][0]
        facts = pd.read_csv(z.open(fact_file), sep=";", dtype=str)

        ref_file = [
            f for f in z.namelist() if "measure_org" in f or "country" in f
        ][0]
        countries = pd.read_csv(z.open(ref_file), sep=";", dtype=str)

    # объединение со справочником стран
    merged = facts.merge(
        countries[["code", "value"]],
        left_on="measure_org",
        right_on="code",
        how="left",
    )
    merged = merged.rename(
        columns={"value": "country", "m_value": metric_name, "period": "date"}
    )

    # приведение типов данных
    merged[metric_name] = pd.to_numeric(merged[metric_name], errors="coerce")
    merged["year"] = pd.to_datetime(merged["date"]).dt.year

    # очистка и дедупликация
    clean_df = merged[["country", "year", metric_name]]
    clean_df = clean_df.dropna(subset=[metric_name])
    clean_df = clean_df.drop_duplicates(subset=["country", "year"])

    return clean_df.sort_values(by=["country", "year"]).reset_index(drop=True)

In [26]:
# Коэффициент младенческой смертности, промилле 

df_infant_mortality = extract_indicator(
    "ts_cis_ft44185", "infant_mortality"
)
display(df_infant_mortality.tail(5))

,country,year,infant_mortality
294,Украина,2017,7.6
295,Украина,2018,7.0
296,Украина,2019,7.0
297,Украина,2020,6.8
298,Украина,2021,7.2


In [27]:
# Сальдо миграции, (тыс. человек)

df_migration = extract_indicator(
      "ts_cis_ft870559", "migration"
  )
display(df_migration.head(5))

,country,year,migration
0,Азербайджан,1991,-40.1
1,Азербайджан,1992,-14.2
2,Азербайджан,1993,-12.2
3,Азербайджан,1994,-11.0
4,Азербайджан,1995,-9.8


In [28]:
# Общий коэффициент естественного прироста, промилле

df_natural_increase = extract_indicator(
      "ts_cis_ft44186", "natural_increase"
  )
display(df_natural_increase.tail(5))

,country,year,natural_increase
323,Украина,2017,-5.1
324,Украина,2018,-6.1
325,Украина,2019,-6.6
326,Украина,2020,-8.1
327,Украина,2021,-11.2


In [46]:
# Общий коэффициент рождаемости, промилле
 
df_fertility_rate = extract_indicator(
      "ts_cis_ft44181", "fertility_rate"
  )
display(df_fertility_rate.head(5))

,country,year,fertility_rate
0,Азербайджан,1991,26.6
1,Азербайджан,1992,25.0
2,Азербайджан,1993,23.7
3,Азербайджан,1994,21.4
4,Азербайджан,1995,18.9


In [44]:
# Общий коэффициент смертности, промилле

df_death_rate = extract_indicator(
      "ts_cis_ft44184", "death_rate"
  )
display(df_death_rate.tail(5))

,country,year,death_rate
327,Украина,2017,14.5
328,Украина,2018,14.8
329,Украина,2019,14.7
330,Украина,2020,15.9
331,Украина,2021,18.5


In [47]:
# Ввод в действие общей площади жилых домов на 1000 населения, квадратных метров

df_home_rate = extract_indicator(
      "ts_cis_ft44007", "home_rate"
  )
display(df_home_rate.tail(5))

,country,year,home_rate
335,Украина,2017,241.0
336,Украина,2018,206.0
337,Украина,2019,264.0
338,Украина,2020,203.3
339,Украина,2021,276.7


In [21]:
# Доля пользователей сети Интернет

df_internet_users = extract_indicator(
      "ts_cis_ft45199", "internet_users"
  )
display(df_internet_users.head(5))

,country,year,internet_users
0,Азербайджан,2005,8.0
1,Азербайджан,2006,10.0
2,Азербайджан,2007,11.0
3,Азербайджан,2008,17.0
4,Азербайджан,2009,27.0


In [48]:
# Использование средств контрацепции, на 1000 женщин в возрасте 15-49 лет

df_using_birth_control = extract_indicator(
      "ts_cis_ft45058", "using_birth_control"
  )
display(df_using_birth_control.head(5))

,country,year,using_birth_control
0,Азербайджан,2018,2370.0
1,Азербайджан,2019,2150.0
2,Азербайджан,2020,1760.0
3,Азербайджан,2021,1493.1
4,Азербайджан,2022,1081.5


In [49]:
# Коэффициент материнской смертности (число женщин, умерших от осложнений беременности, родов и послеродового периода, на 100 000 родившихся живыми)

df_mother_death = extract_indicator(
      "ts_cis_ft4654594", "mother_death"
  )
display(df_mother_death.head(5))

,country,year,mother_death
0,Азербайджан,2000,37.0
1,Азербайджан,2001,25.4
2,Азербайджан,2002,19.9
3,Азербайджан,2003,18.5
4,Азербайджан,2004,25.8


In [51]:
# Ожидаемая продолжительность жизни, лет
df_life_expectancy = extract_indicator(
      "ts_cis_ft44194", "life_expectancy"
  )
display(df_life_expectancy.head(5))

,country,year,life_expectancy
0,Азербайджан,1991,66.3
1,Азербайджан,1992,65.4
2,Азербайджан,1993,65.2
3,Азербайджан,1994,65.2
4,Азербайджан,1995,65.2


In [53]:
# Состоит на учете больных с диагнозом - всего, на 100000 человек населения

df_sick_diagnosis = extract_indicator(
      "ts_cis_ft740623", "sick_diagnosis"
  )
display(df_sick_diagnosis.head(5))

,country,year,sick_diagnosis
0,Азербайджан,2018,88.1
1,Азербайджан,2019,83.0
2,Азербайджан,2020,69.6
3,Азербайджан,2021,66.3
4,Азербайджан,2022,62.6


In [54]:
# Смертность детей в возрасте до 5 лет

df_death_rate_5 = extract_indicator(
      "ts_cis_ft4650801", "death_rate_5"
  )
display(df_death_rate_5.head(5))

,country,year,death_rate_5
0,Азербайджан,2015,13.3
1,Азербайджан,2016,13.8
2,Азербайджан,2017,13.7
3,Азербайджан,2018,13.1
4,Азербайджан,2019,13.0


In [ ]:
# Совокупный коэффициент рождаемости
df_tfr = extract_indicator(
      "ts_cis_ft44190", "tfr"
  )
display(df_tfr.tail(5))

,country,year,tfr
178,Узбекистан,2020,2.900
179,Узбекистан,2021,3.200
180,Узбекистан,2022,3.300
181,Узбекистан,2023,3.400
182,Узбекистан,2024,3.229
183,Украина,2017,1.400
184,Украина,2018,1.300
185,Украина,2019,1.200
186,Украина,2020,1.200
187,Украина,2021,1.200


In [55]:
# Численность среднего медицинского персонала, на 10000 человек населения

df_med_stuff = extract_indicator(
      "ts_cis_ft44786", "med_stuff"
  )
display(df_med_stuff.tail(5))

,country,year,med_stuff
336,Украина,2017,8.0
337,Украина,2018,81.0
338,Украина,2019,78.0
339,Украина,2020,73.0
340,Украина,2021,61.0


In [56]:
# Численность студентов в высших профессиональных учебных учреждениях (на 10000 человек населения), человек

df_high_educ = extract_indicator(
      "ts_cis_ft720150", "high_educ"
  )
display(df_high_educ.tail(5))

,country,year,high_educ
345,Украина,2017,314.0
346,Украина,2018,314.0
347,Украина,2019,302.0
348,Украина,2020,274.0
349,Украина,2021,254.0


In [57]:
# Число абонентов мобильного широкополосного доступа к сети Интернет на 1000 населения, единиц

df_mb_subs= extract_indicator(
      "ts_cis_ft45198", "Mobile_broadband_subs"
  )
display(df_mb_subs.tail(5))

,country,year,Mobile_broadband_subs
26,Россия,2022,1092.000000
27,Россия,2023,1158.895851
28,Узбекистан,2021,588.700000
29,Узбекистан,2022,660.900000
30,Узбекистан,2023,739.164845


In [58]:
# Число абортов у женщин, на 1000 женщин в возрасте 15-49 лет

df_abortions = extract_indicator(
      "ts_cis_ft45056", "abortions"
  )
display(df_abortions.tail(5))

,country,year,abortions
55,Узбекистан,2020,1.200000
56,Узбекистан,2021,0.700000
57,Узбекистан,2022,0.200000
58,Узбекистан,2023,0.200000
59,Узбекистан,2024,0.000212


In [59]:
# Число больничных организаций, на 10000 человек населения

df_hospital = extract_indicator(
      "ts_cis_ft44788", "hospital"
  )
display(df_hospital.head(5))

,country,year,hospital
0,Азербайджан,2010,0.8
1,Азербайджан,2011,0.6
2,Азербайджан,2012,0.6
3,Азербайджан,2013,0.6
4,Азербайджан,2014,0.6


In [60]:
# Число браков на 1000 человек

df_marriages = extract_indicator(
      "ts_cis_ft44187", "marriages"
  )
display(df_marriages.head(5))

,country,year,marriages
0,Азербайджан,1991,10.4
1,Азербайджан,1992,9.5
2,Азербайджан,1993,8.1
3,Азербайджан,1994,6.3
4,Азербайджан,1995,5.7


In [62]:
# Число пострадавших со смертельным исходом, на 10000 работающих

df_fatalities= extract_indicator(
      "ts_cis_ft45171", "fatalities"
  )
display(df_fatalities.head(5))

,country,year,fatalities
0,Азербайджан,1991,1.0
1,Азербайджан,1992,1.0
2,Азербайджан,1993,1.0
3,Азербайджан,1994,1.0
4,Азербайджан,1995,1.0


In [63]:
# Число построенных квартир на 10000 населения, единиц

df_apartments_built= extract_indicator(
      "ts_cis_ft44009", "apartments_built"
  )
display(df_apartments_built.head(5))

,country,year,apartments_built
0,Азербайджан,1991,40.0
1,Азербайджан,1992,33.0
2,Азербайджан,1993,23.0
3,Азербайджан,1994,12.0
4,Азербайджан,1995,9.0


In [64]:
# Уровень занятости, проценты
df_empl = extract_indicator(
      "ts_cis_ft43228", "Employment"
  )
display(df_empl.head(5))

,country,year,Employment
0,Азербайджан,2010,29.2
1,Азербайджан,2011,30.2
2,Азербайджан,2012,31.7
3,Азербайджан,2013,33.5
4,Азербайджан,2014,66.1


In [69]:
# Уровень безработицы, проценты
df_unempl = extract_indicator(
      "ts_cis_ft43229", "Unemployment"
  )
display(df_unempl.head(5))

,country,year,Unemployment
0,Азербайджан,2010,14.9
1,Азербайджан,2011,14.7
2,Азербайджан,2012,14.2
3,Азербайджан,2013,13.7
4,Азербайджан,2014,13.5


In [70]:
# Число разводов на 1000 человек

df_divorces = extract_indicator(
      "ts_cis_ft44188", "divorces"
  )
display(df_divorces.head(5))

,country,year,divorces
0,Азербайджан,1991,1.5
1,Азербайджан,1992,1.3
2,Азербайджан,1993,0.9
3,Азербайджан,1994,0.8
4,Азербайджан,1995,0.8


In [75]:
# Индекс потребительских цен

df_cpi = extract_indicator(
      "ts_cis_ft43370", "cpi"
  )
display(df_cpi.tail(5))

,country,year,cpi
366,Украина,2017,113.7
367,Украина,2018,101.5
368,Украина,2019,101.0
369,Украина,2020,100.2
370,Украина,2021,101.3


In [77]:
# Число посещений музеев в расчете на 1000 человек населения,единиц

df_museum_visits = extract_indicator(
      "ts_cis_ft720178", "museum_visits"
  )
display(df_museum_visits.tail(5))

,country,year,museum_visits
321,Украина,2013,492.0
322,Украина,2014,332.0
323,Украина,2015,353.0
324,Украина,2016,371.0
325,Украина,2017,387.0


In [78]:
# Число посещений театров в расчете на 1000 человек населения,единиц

df_theaters_visits = extract_indicator(
      "ts_cis_ft720175", "theaters_visits"
  )
display(df_theaters_visits.tail(5))

,country,year,theaters_visits
310,Украина,2013,151.0
311,Украина,2014,127.0
312,Украина,2015,132.0
313,Украина,2016,140.0
314,Украина,2017,146.0


In [79]:
from functools import reduce
import pandas as pd

# список показателей Статкомитета СНГ
table_names = [
    "df_infant_mortality",
    "df_migration",
    "df_natural_increase",
    "df_fertility_rate",
    "df_death_rate",
    "df_internet_users",
    "df_using_birth_control",
    "df_mother_death",
    "df_life_expectancy",
    "df_sick_diagnosis",
    "df_death_rate_5",
    "df_tfr",
    "df_med_stuff",
    "df_high_educ",
    "df_mb_subs",
    "df_abortions",
    "df_hospital",
    "df_marriages",
    "df_fatalities",
    "df_apartments_built",
    "df_empl",
    "df_unempl",
    "df_divorces",
    "df_cpi",
    "df_museum_visits",
    "df_theaters_visits"
]

# подготовка таблиц к объединению
dfs_to_merge = []
for name in table_names:
    if name in globals():
        df = globals()[name].copy()
        df = df.drop_duplicates(subset=["country", "year"])
        df["country"] = df["country"].astype(str)
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df = df.dropna(subset=["year"])
        df["year"] = df["year"].astype(int)
        dfs_to_merge.append(df)

# объединение всех витрин по ключам country и year
df_master_cis = reduce(
    lambda left, right: pd.merge(
        left, right, on=["country", "year"], how="outer"
    ),
    dfs_to_merge,
)

# очистка и сортировка
df_master_cis = df_master_cis[
    df_master_cis["country"].ne("nan") & df_master_cis["country"].notna()
]
df_master_cis = df_master_cis.sort_values(
    by=["country", "year"]
).reset_index(drop=True)

print(
    f"Итоговый датасет: {df_master_cis.shape[0]} строк, {df_master_cis.shape[1]} колонок."
)
display(df_master_cis.head())

Итоговый датасет: 380 строк, 27 колонок.


,country,year,infant_mortality,migration,natural_increase,fertility_rate,death_rate,internet_users,using_birth_control,mother_death,...,hospital,marriages,fatalities,apartments_built,Employment,Unemployment,divorces,cpi,museum_visits,theaters_visits
0,Азербайджан,1991,25.3,-40.1,20.4,26.6,6.2,NaN,NaN,NaN,...,NaN,10.4,1.0,40.0,NaN,NaN,1.5,329.7,357.0,218.0
1,Азербайджан,1992,25.5,-14.2,17.9,25.0,7.1,NaN,NaN,NaN,...,NaN,9.5,1.0,33.0,NaN,NaN,1.3,1672.4,275.0,186.0
2,Азербайджан,1993,28.2,-12.2,16.5,23.7,7.2,NaN,NaN,NaN,...,NaN,8.1,1.0,23.0,NaN,NaN,0.9,1392.5,306.0,212.0
3,Азербайджан,1994,25.2,-11.0,14.1,21.4,7.3,NaN,NaN,NaN,...,NaN,6.3,1.0,12.0,NaN,NaN,0.8,1886.7,210.0,192.0
4,Азербайджан,1995,23.3,-9.8,12.2,18.9,6.7,NaN,NaN,NaN,...,NaN,5.7,1.0,9.0,NaN,NaN,0.8,184.6,204.0,159.0


In [80]:
csv_path = "cis_dataset.csv"
excel_path = "cis_dataset.xlsx"

df_master_cis.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_master_cis.to_excel(excel_path, index=False, sheet_name="CIS_Data")